In [1]:
import scanpy as sc
import scvi
import numpy as np
import sys
sys.path.append('../')

from scripts.subset_hvg import subset_to_hvg

adata = sc.read_h5ad(
    "../../data/obesity_subset_CEBPB_KIF11.h5ad"
)

print(adata)

adata_subset, hvg_genes, sig_genes = subset_to_hvg(
    adata,
    include_signature_genes=True
)


print(adata_subset)

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnnData object with n_obs × n_vars = 24325 × 36601
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_guide', 'nFeature_guide', 'percent.mt', 'SampleID', 'Day', 'num_features', 'feature_call', 'num_umis', 'gene', 'adipo', 'pre_adipo', 'other', 'lipo'
    uns: 'log1p'
    layers: 'counts'
HVG requested: 5000
Signature genes requested: 820
Signature genes found in dataset: 797
Total genes used: 5563
Missing genes: 0
AnnData object with n_obs × n_vars = 24325 × 5563
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_guide', 'nFeature_guide', 'percent.mt', 'SampleID', 'Day', 'num_features', 'feature_call', 'num_umis', 'gene', 'adipo', 'pre_adipo', 'other', 'lipo'
    uns: 'log1p'
    layers: 'counts'


In [2]:
import numpy as np

def create_scanvi_labels(adata):

    labels = []

    for _, row in adata.obs.iterrows():

        if row["lipo"] == 1:
            labels.append("lipo")

        elif row["adipo"] == 1:
            labels.append("adipo")

        elif row["pre_adipo"] == 1:
            labels.append("pre_adipo")

        else:
            labels.append("other")

    adata.obs["cell_state"] = labels

    return adata


adata_subset = create_scanvi_labels(adata_subset)

print(adata_subset.obs["cell_state"].value_counts())

cell_state
other        12084
pre_adipo     7379
adipo         3824
lipo          1038
Name: count, dtype: int64


In [6]:
train_perts = [
    "NC",
    "NC+NC",
    "CEBPB",
    "KIF11",
    "CEBPB+NC",
    "KIF11+NC"
]

adata_train = adata_subset[
    adata_subset.obs["gene"].isin(train_perts)
].copy()

adata_test = adata_subset[
    adata_subset.obs["gene"] == "CEBPB+KIF11"
].copy()

In [ ]:
import scvi

# setup anndata
scvi.model.SCVI.setup_anndata(
    adata_train,
    layer="counts"
)

# 先訓練 unsupervised VAE
vae = scvi.model.SCVI(
    adata_train,
    n_latent=50
)

vae.train(
    max_epochs=200,
    accelerator="mps",
    devices=1,
    batch_size=256,
    early_stopping=True
)

# 再轉成 semi-supervised model
scanvi = scvi.model.SCANVI.from_scvi_model(
    vae,
    labels_key="cell_state",
    unlabeled_category="unknown"   # 必須是字串
)

scanvi.train(
    max_epochs=200,
    accelerator="mps",
    devices=1,
    batch_size=256,
    early_stopping=True
)

# 取得 latent embedding
adata_train.obsm["X_scanvi"] = scanvi.get_latent_representation(adata_train)

print(adata_train.obsm["X_scanvi"].shape)

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/anaconda3/envs

Epoch 200/200: 100%|██████████| 200/200 [11:12<00:00,  3.27s/it, v_num=1, train_loss=3.79e+3]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [11:12<00:00,  3.36s/it, v_num=1, train_loss=3.79e+3]


ValueError: Categorical categories cannot be null

In [ ]:

# 再轉成 semi-supervised model
scanvi = scvi.model.SCANVI.from_scvi_model(
    vae,
    labels_key="cell_state",
    unlabeled_category="unknown"   # 必須是字串
)

scanvi.train(
    max_epochs=200,
    accelerator="mps",
    devices=1,
    batch_size=256,
    early_stopping=True
)

# 取得 latent embedding
adata_train.obsm["X_scanvi"] = scanvi.get_latent_representation(adata_train)

print(adata_train.obsm["X_scanvi"].shape)

INFO     Training for 200 epochs.                                                                                  


/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/anaconda3/envs

Epoch 27/200:  13%|█▎        | 26/200 [03:53<24:08,  8.32s/it, v_num=1, train_loss=3.77e+3]

In [ ]:
save_dir = "model/scanvi_model"

scanvi.save(
    save_dir,
    overwrite=True
)

print("model saved to:", save_dir)

In [ ]:
# load model
scanvi = scvi.model.SCANVI.load(
    "model/scanvi_model",
    adata=adata_train
)

# query data mapping
adata_test = scvi.model.SCANVI.prepare_query_anndata(
    adata_test,
    scanvi
)

# latent representation
adata_test.obsm["X_scanvi"] = scanvi.get_latent_representation(
    adata_test
)

print(adata_test.obsm["X_scanvi"].shape)

In [ ]:
adata_train.obsm["X_scanvi"], adata_test.obsm["X_scanvi"]

In [ ]:
from sklearn.model_selection import train_test_split

X = adata_train.obsm["X_scanvi"]
y = adata_train.obs["cell_state"]

X_train, X_val, y_train, y_val = train_test_split(

    X,
    y,

    test_size=0.2,

    stratify=y,

    random_state=42
)

from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


clf = LGBMClassifier(

    n_estimators=500,

    learning_rate=0.05,

    num_leaves=64,

    random_state=42
)

clf.fit(

    X_train,

    y_train,

    eval_set=[(X_val, y_val)],

    eval_metric="multi_logloss"
)

cell
P1_AAACGAATCCTGGTCC-1      TF15_MOI2_1
P1_AAACGATGTAGCAGAC-1      TF15_MOI2_1
P1_AAACGATGTTAGGCCC-1      TF15_MOI2_1
P1_AAACGATGTTATGCGT-1      TF15_MOI2_1
P1_AAACGATGTTGTAGCT-1      TF15_MOI2_1
                              ...     
P12_TGTGCCCTCGACATGC-1    TF15_MOI2_12
P12_TGTGGTTGTTAACGTC-1    TF15_MOI2_12
P12_TGTGTACGTCATGTCG-1    TF15_MOI2_12
P12_TGTGTTAGTCTGGTCA-1    TF15_MOI2_12
P12_TGTGTTAGTTATCCAA-1    TF15_MOI2_12
Name: batch, Length: 24283, dtype: object

In [ ]:
y_pred = clf.predict(X_val)

print("VAL accuracy:")
print(accuracy_score(y_val, y_pred))

print("\nclassification report")
print(classification_report(y_val, y_pred))

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/anaconda3/envs

Epoch 200/200: 100%|██████████| 200/200 [07:03<00:00,  2.19s/it, v_num=1, train_loss=2.43e+3]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [07:03<00:00,  2.12s/it, v_num=1, train_loss=2.43e+3]


In [ ]:
test_pred = clf.predict(

    adata_test.obsm["X_scanvi"]
)

adata_test.obs["pred_cell_state"] = test_pred

print(

    adata_test.obs["pred_cell_state"].value_counts()

)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import pandas as pd


y_true = adata_test.obs["cell_state"]
y_pred = adata_test.obs["pred_cell_state"]

print("Accuracy")
print(accuracy_score(y_true, y_pred))

print("\nClassification report")
print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

print("\nConfusion matrix (row-normalized)")
print(
    pd.crosstab(
        y_true,
        y_pred,
        normalize="index"
    )
)

In [ ]:
import numpy as np


def compute_proportion_df(adata, label_col):

    rows = []

    for g in adata.obs["gene"].unique():

        subset = adata.obs[
            adata.obs["gene"] == g
        ]

        result = {

            "gene": g,

            "pre_adipo":
                (subset[label_col] == "pre_adipo").mean(),

            "adipo":
                (subset[label_col] == "adipo").mean(),

            "lipo":
                (subset[label_col] == "lipo").mean(),

            "other":
                (subset[label_col] == "other").mean(),
        }

        result["lipo_adipo"] = (

            result["lipo"] /

            (result["adipo"] + 1e-20)

        )

        rows.append(result)

    return pd.DataFrame(rows)

In [ ]:
def compute_metric_l1_distance(
    true_state_proportion_df: pd.DataFrame,
    pred_state_proprotion_df: pd.DataFrame,
) -> float:
    # Going over all the genes that were perturbed in this set
    unique_perturb_genes = list(true_state_proportion_df["gene"].unique())

    all_l1_loss_list = []
    for gene in unique_perturb_genes:
        # Slicing the column with this gene
        true_gene_df = true_state_proportion_df[true_state_proportion_df["gene"] == gene]
        pred_gene_df = pred_state_proprotion_df[pred_state_proprotion_df["gene"] == gene]

        # print(gene, pred_gene_df.shape[0])
        assert true_gene_df.shape[0] == 1 and pred_gene_df.shape[0] == 1, f"Invalid prediction count for state gene={gene} count={pred_gene_df.shape[0]}!=1"

        # Getting the L1 loss for main  pre, adipo and other
        l1_three = (
            np.abs(true_gene_df.iloc[0]["pre_adipo"] - pred_gene_df.iloc[0]["pre_adipo"]) +
            np.abs(true_gene_df.iloc[0]["adipo"] - pred_gene_df.iloc[0]["adipo"]) +
            np.abs(true_gene_df.iloc[0]["other"] - pred_gene_df.iloc[0]["other"])
        )

        # Getting the L1 loss for lipo by adipo
        numerical_stab_term = 1e-20
        pred_lipo_adipo = pred_gene_df.iloc[0]["lipo"] / (pred_gene_df.iloc[0]["adipo"] + numerical_stab_term)
        true_lipo_adipo = true_gene_df.iloc[0]["lipo"] / (true_gene_df.iloc[0]["adipo"] + numerical_stab_term)
        l1_lipo_adipo = np.abs(true_lipo_adipo - pred_lipo_adipo)

        # Getting the average error
        average_l1 = 0.75 * l1_three + 0.25 * l1_lipo_adipo
        all_l1_loss_list.append(average_l1)

    # Getting the overall average over all the gene perturbation
    l1_loss = np.mean(all_l1_loss_list)
    return float(l1_loss)

In [ ]:
gtruth_proportion = compute_proportion_df(

    adata_test,

    "cell_state"

)

predicted_proportion = compute_proportion_df(

    adata_test,

    "pred_cell_state"

)
l1_distance = compute_metric_l1_distance(

    gtruth_proportion,

    predicted_proportion,

)

print(f"L1-distance score: {l1_distance:.4f}")

,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,gene
cell,,,,,,,,,,,,,,,,,,,,,
P1_AAACGAATCCTGGTCC-1,0.585291,-1.280644,1.597075,-0.883349,0.736986,1.107108,0.902766,-0.437251,-1.057230,-0.273204,...,-0.102122,1.538182,1.656824,-0.605873,-0.407191,0.653887,1.420594,0.373635,0.204760,CEBPB
P1_AAACGATGTAGCAGAC-1,-0.162557,0.890362,1.405463,-0.081123,1.916624,-0.032016,-0.594495,-0.660733,0.562375,-1.085142,...,-2.610894,-1.005070,1.120222,-1.047677,-0.826163,-2.181262,0.030551,0.615936,-0.240628,NC
P1_AAACGATGTTAGGCCC-1,-0.143628,-0.043459,-0.531773,-0.604031,-1.256630,0.558056,0.364586,-2.099229,0.283861,0.284595,...,0.216992,0.635148,-1.815317,0.529265,0.385391,-0.058370,1.361046,0.349199,0.492675,NC
P1_AAACGATGTTATGCGT-1,-0.608182,-0.731126,0.342763,1.450643,1.608000,-0.520168,0.336813,0.615886,-1.008046,-1.293193,...,-0.245519,-0.927297,0.689492,1.471456,0.686037,-1.145534,-0.532311,-0.797233,0.550980,NC
P1_AAACGATGTTGTAGCT-1,0.168928,1.454000,0.274039,-0.069807,-1.553495,0.171298,1.189952,0.732102,-0.821019,-0.286427,...,-0.244276,-1.744291,0.232527,0.273635,-0.023589,-0.134384,0.220542,-0.600689,-1.295704,NC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P12_TGTGCCCTCGACATGC-1,1.766234,0.377626,0.888005,0.553132,-0.276341,-0.058823,0.974348,-0.797711,0.940204,-0.769217,...,-0.015837,-0.097602,-1.753697,-1.923464,-0.298857,-0.883677,-1.047282,0.255984,0.797474,NC
P12_TGTGGTTGTTAACGTC-1,-0.985781,-1.363168,0.050506,0.425575,-0.823969,0.050892,-0.362249,1.014184,0.324753,-0.030067,...,-0.333400,-2.135866,1.329627,1.058062,0.718560,0.298259,-0.477554,-0.580132,-0.057337,NC
P12_TGTGTACGTCATGTCG-1,0.258406,0.492646,0.344593,0.116795,-0.020474,0.880835,0.527286,-1.646245,0.767315,1.682338,...,0.221983,0.572483,-0.550084,0.725393,0.103156,0.213967,-0.022456,-0.339688,2.082110,NC
